# 05 — Deployment Models and Dashboard Data

Builds the models and data behind the prediction dashboard. Two logistic-regression models are trained in an expanding-window (walk-forward) setup — one on seasons up to 2023-24 to predict 2024-25, and one on seasons up to 2024-25 to predict 2025-26 — keeping every prediction leakage-free. Per-game predictions, factor contributions, and a season accuracy tally are exported for the Streamlit app, alongside the two saved models. The dashboard demonstrates the pipeline end to end; the accuracy shown there is illustrative rather than a headline result.

## Setup and configuration

Paths, the season lists for the walk-forward design, and the column conventions from NB01. The
deploy-only `era_of` extends NB01's early/modern tags to the newly collected middle seasons
(`mid`) and the new season (`new`); era is metadata only — the models select by `Season`.

In [1]:
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from nba_api.stats.endpoints.leaguegamelog import LeagueGameLog

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Seasons needed for the deployment models (early 2007-12 block deliberately excluded).
MIDDLE_SEASONS = ["2012-13", "2013-14", "2014-15", "2015-16",
                  "2016-17", "2017-18", "2018-19", "2019-20"]
NEW_SEASONS = ["2025-26"]

TRAIN_A = MIDDLE_SEASONS + ["2020-21", "2021-22", "2022-23", "2023-24"]
PREDICT_A = "2024-25"
TRAIN_B = TRAIN_A + [PREDICT_A]
PREDICT_B = "2025-26"


def era_of(season):
    year = int(season[:4])
    if 2007 <= year <= 2011:
        return "early"
    if 2012 <= year <= 2019:
        return "mid"
    if 2020 <= year <= 2024:
        return "modern"
    return "new"


RENAME = {
    "GAME_ID": "GAME_ID", "GAME_DATE": "Date", "TEAM_ABBREVIATION": "Team", "WL": "WL",
    "FGM": "FGM", "FGA": "FGA", "FG3M": "FG3M", "FG3A": "FG3A", "FTM": "FTM", "FTA": "FTA",
    "OREB": "ORB", "DREB": "DRB", "TOV": "TOV", "PTS": "PTS",
}
FINAL_COLS = [
    "GAME_ID", "Date", "Season", "Team", "Opp", "Home", "W",
    "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA", "TOV", "PTS", "ORB", "DRB", "era",
]

## Step 0 — Collect the missing seasons

The thesis raw data only holds the two era blocks (2007-12, 2020-25). The dashboard needs the
missing middle seasons (2012-13 -> 2019-20) and the new completed season (2025-26). Collect them
with the exact NB01 logic (same columns, two-rows-per-game, retry/backoff). Seasons already on
disk are skipped, so this is cheap on re-run.

In [2]:
def get_opponent(matchup):
    separator = " vs. " if "vs." in matchup else " @ "
    return matchup.split(separator)[1].strip()


def fetch_season(season):
    for wait in [0, 5, 15, 30]:
        if wait:
            time.sleep(wait)
        try:
            log = LeagueGameLog(player_or_team_abbreviation="T",
                                season_type_all_star="Regular Season",
                                season=season, timeout=60)
            return log.get_data_frames()[0]
        except Exception:
            continue
    raise RuntimeError(f"Could not download {season} after several attempts")


def clean(raw, season):
    df = raw[list(RENAME)].rename(columns=RENAME)
    df["Date"] = pd.to_datetime(df["Date"])
    df["Season"] = season
    df["era"] = era_of(season)
    df["W"] = (df["WL"] == "W").astype(int)
    df["Opp"] = raw["MATCHUP"].apply(get_opponent)
    df["Home"] = raw["MATCHUP"].apply(lambda m: 1 if "vs." in m else 0)
    return df[FINAL_COLS]


for season in MIDDLE_SEASONS + NEW_SEASONS:
    out_path = RAW_DIR / f"team_games_{season}.csv"
    if out_path.exists():
        print(f"{season}: already saved")
        continue
    print(f"{season}: downloading...")
    clean(fetch_season(season), season).to_csv(out_path, index=False)
    time.sleep(1.5 + random.uniform(0, 1.5))
    print(f"{season}: saved")

2012-13: already saved
2013-14: already saved
2014-15: already saved
2015-16: already saved
2016-17: already saved
2017-18: already saved
2018-19: already saved
2019-20: already saved
2025-26: already saved


## Step 1 — Build the deployment feature matrix

Re-run the NB02 feature pipeline over the full collected span so every season gets identical
`home_/away_/diff_` rolling Four Factors features (10-game rolling, `shift(1)`, `min_periods=5`,
grouped by Team+Season). The code below is NB02's transformation, applied to all raw seasons on
disk and written to a **separate** deploy matrix so the thesis matrix is untouched. ORB% uses the
opponent's DRB via the same-game self-join, exactly as NB02.

In [3]:
# Load every per-season raw file (exclude the combined thesis file).
season_files = sorted(p for p in RAW_DIR.glob("team_games_*.csv") if p.name != "team_games_all.csv")
games = pd.concat(
    [pd.read_csv(p, dtype={"GAME_ID": str}, parse_dates=["Date"]) for p in season_files],
    ignore_index=True,
)

# --- NB02 feature engineering (identical computation) ---
games["opp_DRB"] = games.groupby("GAME_ID")["DRB"].transform("sum") - games["DRB"]
games["eFG"] = (games["FGM"] + 0.5 * games["FG3M"]) / games["FGA"]
games["TOV_pct"] = games["TOV"] / (games["FGA"] + 0.44 * games["FTA"] + games["TOV"])
games["ORB_pct"] = games["ORB"] / (games["ORB"] + games["opp_DRB"])
games["FTrate"] = games["FTM"] / games["FGA"]

FACTORS = ["eFG", "TOV_pct", "ORB_pct", "FTrate"]

games["pace"] = games["FGA"] + 0.44 * games["FTA"] - games["ORB"] + games["TOV"]
games = games.sort_values(["Team", "Season", "Date"])
games["rest"] = games.groupby(["Team", "Season"])["Date"].diff().dt.days

for col in FACTORS + ["pace"]:
    games[col + "_roll"] = (
        games.groupby(["Team", "Season"])[col]
        .transform(lambda s: s.shift(1).rolling(10, min_periods=5).mean())
    )

SIDE_MAP = {
    "eFG_roll": "eFG", "TOV_pct_roll": "TOV_pct", "ORB_pct_roll": "ORB_pct",
    "FTrate_roll": "FTrate", "pace_roll": "pace", "rest": "rest",
}

home = games[games["Home"] == 1]
away = games[games["Home"] == 0]

home_feats = home[["GAME_ID"] + list(SIDE_MAP)].rename(
    columns={col: "home_" + name for col, name in SIDE_MAP.items()})
away_feats = away[["GAME_ID"] + list(SIDE_MAP)].rename(
    columns={col: "away_" + name for col, name in SIDE_MAP.items()})

# Identifiers + target + home/away team names (names needed for the dashboard dropdown).
ids = home[["GAME_ID", "Date", "Season", "era", "W", "Team", "Opp"]].rename(
    columns={"W": "y", "Team": "home_team", "Opp": "away_team"})

matrix = ids.merge(home_feats, on="GAME_ID").merge(away_feats, on="GAME_ID")
for name in SIDE_MAP.values():
    matrix["diff_" + name] = matrix["home_" + name] - matrix["away_" + name]

id_cols = ["GAME_ID", "Date", "Season", "era", "y", "home_team", "away_team"]
four_factor_cols = [f"{side}_{f}" for f in FACTORS for side in ["home", "away", "diff"]]
challenger_cols = [f"{side}_{c}" for c in ["pace", "rest"] for side in ["home", "away", "diff"]]
feature_cols = four_factor_cols + challenger_cols

matrix = matrix[id_cols + feature_cols].sort_values("Date").reset_index(drop=True)
matrix = matrix.dropna(subset=feature_cols).reset_index(drop=True)

matrix.to_csv(PROCESSED_DIR / "team_games_features_deploy.csv", index=False)
matrix.groupby("Season").size()

Season
2007-08    1150
2008-09    1152
2009-10    1149
2010-11    1149
2011-12     909
2012-13    1150
2013-14    1153
2014-15    1151
2015-16    1152
2016-17    1151
2017-18    1150
2018-19    1151
2019-20     982
2020-21    1000
2021-22    1152
2022-23    1151
2023-24    1149
2024-25    1148
2025-26    1146
dtype: int64

## Step 2 — Train the two walk-forward models

Same model as NB03: logistic regression on the 12 core Four Factors features, `StandardScaler`
in a pipeline, no tuning. Model A trains strictly before 2024-25; Model B strictly before
2025-26. A hard assertion verifies the target season has zero rows in each training frame
(non-negotiable leakage guard) before fitting. Models are saved to `joblib`.

In [4]:
FEATURES = [f"{side}_{f}" for f in FACTORS for side in ["home", "away", "diff"]]


def make_logreg():
    return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])


def train_model(train_seasons, target_season):
    train = matrix[matrix["Season"].isin(train_seasons)]
    # Leakage guard: the predicted season must never appear in training.
    assert (train["Season"] == target_season).sum() == 0, "LEAKAGE: target season in training data"
    model = make_logreg().fit(train[FEATURES], train["y"])
    return model


model_a = train_model(TRAIN_A, PREDICT_A)
model_b = train_model(TRAIN_B, PREDICT_B)

joblib.dump(model_a, MODEL_DIR / "model_a_2024-25.joblib")
joblib.dump(model_b, MODEL_DIR / "model_b_2025-26.joblib")
print("Model A trained on", TRAIN_A[0], "->", TRAIN_A[-1], "| predicts", PREDICT_A)
print("Model B trained on", TRAIN_B[0], "->", TRAIN_B[-1], "| predicts", PREDICT_B)

Model A trained on 2012-13 -> 2023-24 | predicts 2024-25
Model B trained on 2012-13 -> 2024-25 | predicts 2025-26


## Step 3 — Precompute per-game predictions and factor contributions

For every game in each target season, store the predicted home-win probability, the predicted
label (`p >= 0.5`), the actual result, and the four per-factor contributions. Each contribution
is the standardized coefficient of that factor's `diff_` feature times the game's standardized
`diff_` value, so a positive value pushed the prediction toward the home team and a negative
value toward the away team. (This is an interpretability view of the differential terms, not a
full decomposition of the probability.) Saved to one CSV per season.

In [5]:
def factor_contributions(model, X):
    # Standardized-coef x standardized diff-value for each factor's differential feature.
    scaler = model.named_steps["scale"]
    coef = model.named_steps["clf"].coef_[0]
    out = {}
    for f in FACTORS:
        j = FEATURES.index(f"diff_{f}")
        std_val = (X[f"diff_{f}"].to_numpy() - scaler.mean_[j]) / scaler.scale_[j]
        out[f"contrib_{f}"] = coef[j] * std_val
    return pd.DataFrame(out, index=X.index)


def precompute(model, target_season):
    games_t = matrix[matrix["Season"] == target_season].copy()
    X = games_t[FEATURES]
    p = model.predict_proba(X)[:, 1]
    out = games_t[["GAME_ID", "Date", "Season", "home_team", "away_team", "y"]].copy()
    out = out.rename(columns={"y": "actual_home_win"})
    out["p_home_win"] = p.round(4)
    out["pred_home_win"] = (p >= 0.5).astype(int)
    out["correct"] = (out["pred_home_win"] == out["actual_home_win"]).astype(int)
    contribs = factor_contributions(model, X).round(4)
    out = pd.concat([out.reset_index(drop=True), contribs.reset_index(drop=True)], axis=1)
    return out


pred_a = precompute(model_a, PREDICT_A)
pred_b = precompute(model_b, PREDICT_B)

pred_a.to_csv(PROCESSED_DIR / f"05_predictions_{PREDICT_A}.csv", index=False)
pred_b.to_csv(PROCESSED_DIR / f"05_predictions_{PREDICT_B}.csv", index=False)
print(f"{PREDICT_A}: {len(pred_a)} games | {PREDICT_B}: {len(pred_b)} games")

2024-25: 1148 games | 2025-26: 1146 games


## Step 4 — Per-season accuracy tally

Model accuracy (predicted correct / total) for each season next to the always-pick-home
baseline, for honest comparison. Saved to `05_tally.csv` and used by the dashboard's tally
panel.

In [6]:
tally_rows = []
for preds, season in [(pred_a, PREDICT_A), (pred_b, PREDICT_B)]:
    n = len(preds)
    correct = int(preds["correct"].sum())
    home_base = preds["actual_home_win"].mean()
    tally_rows.append({
        "season": season,
        "n_games": n,
        "model_correct": correct,
        "model_accuracy": round(correct / n, 4),
        "always_home_accuracy": round(home_base, 4),
    })

tally = pd.DataFrame(tally_rows)
tally.to_csv(PROCESSED_DIR / "05_tally.csv", index=False)
tally

,season,n_games,model_correct,model_accuracy,always_home_accuracy
0,2024-25,1148,717,0.6246,0.5497
1,2025-26,1146,684,0.5969,0.5515
